## CNN on Image Data

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## CNN Architecture

In [2]:
class HouseCNN(nn.Module):
    """
    CNN for image classification.
    Can classify house exterior photos into price tiers.
    """
    def __init__(self, num_classes=3):
        super(HouseCNN, self).__init__()

        # Feature extraction — convolutional layers
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 3 channels in, 32 out
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2),                           # halve spatial dims
            nn.Dropout2d(0.1),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 32 in, 64 out
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),                           # halve again
            nn.Dropout2d(0.1),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 64 in, 128 out
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2),
        )

        # Classification — fully connected layers
        self.classifier = nn.Sequential(
            nn.Flatten(),                    # 3D → 1D
            nn.Linear(128 * 4 * 4, 256),    # depends on input size
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)      # final output
        )

    def forward(self, x):
        x = self.features(x)        # extract spatial features
        x = self.classifier(x)      # classify
        return x

model_cnn = HouseCNN(num_classes=3).to(device)

# Count parameters
params = sum(p.numel() for p in model_cnn.parameters())
print(f"CNN Parameters: {params:,}")

CNN Parameters: 619,011


### What padding=1 does?

# padding=1 with 3×3 filter keeps output same size as input
# Without padding: 32×32 → 30×30 (shrinks by 2 each conv)
# With padding=1:  32×32 → 32×32 (same size maintained)
# This is called 'same' padding

## CNN Training on CIFAR-10 (real runnable code)

In [3]:
# Load CIFAR-10 — 60,000 32×32 color images, 10 classes
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))  # normalize each channel
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=64, shuffle=False)

# Adjust classifier for CIFAR-10 input size 32×32
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16×16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 8×8
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2) # 4×4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

cnn = CIFAR10CNN().to(device)
optimizer = optim.Adam(cnn.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(10):
    cnn.train()
    running_loss = 0
    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cnn(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    cnn.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = cnn(images)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    print(f"Epoch {epoch+1:2d} | Loss: {running_loss/len(trainloader):.3f} "
          f"| Test Acc: {100*correct/total:.1f}%")

100%|██████████| 170M/170M [1:14:39<00:00, 38.1kB/s]


Epoch  1 | Loss: 1.483 | Test Acc: 60.5%
Epoch  2 | Loss: 1.077 | Test Acc: 66.4%
Epoch  3 | Loss: 0.901 | Test Acc: 71.4%
Epoch  4 | Loss: 0.790 | Test Acc: 73.9%
Epoch  5 | Loss: 0.705 | Test Acc: 75.1%
Epoch  6 | Loss: 0.633 | Test Acc: 75.2%
Epoch  7 | Loss: 0.580 | Test Acc: 76.6%
Epoch  8 | Loss: 0.520 | Test Acc: 76.0%
Epoch  9 | Loss: 0.472 | Test Acc: 76.0%
Epoch 10 | Loss: 0.438 | Test Acc: 77.0%
